In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



ImportError: DLL load failed while importing _imaging: The specified module could not be found.

In [42]:
# Load data set and its info
df = pd.read_csv("toxicity dataset.csv")
df. head()
df.shape
df.columns
df.info()
x = df.drop("Class", axis =1)
y=df["Class"]
(x.var() == 0).sum()
x.var().sort_values().head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Columns: 1204 entries, MATS3v to Class
dtypes: float64(1003), int64(200), object(1)
memory usage: 1.6+ MB


ETA_dAlpha_A    6.570760e-07
AATSC4c         2.031838e-06
AATSC3c         2.155254e-06
AATSC2c         2.379464e-06
AATSC5c         3.050667e-06
AVP-7           3.117846e-06
AATSC7c         3.292758e-06
JGI10           3.295571e-06
AATSC6c         3.381384e-06
JGI9            3.909966e-06
dtype: float64

In [43]:
df.describe()

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,...,WTPT-3,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22
count,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,...,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000,171.000000
mean,-0.031244,0.315789,-0.001001,-0.061501,0.994152,0.677770,-0.025418,-0.053289,1.569251,0.003226,...,23.229975,8.134013,13.490291,0.202529,1.235093,0.011316,1.467836,0.004820,-0.085088,0.061702
std,0.063559,0.762918,0.063928,0.072891,1.108773,1.647322,0.078645,0.109463,2.497362,0.074076,...,6.440623,4.436831,6.229089,0.024356,0.137024,0.005482,1.013361,0.044475,0.029273,0.241896
min,-0.311500,0.000000,-0.184600,-0.348500,0.000000,0.000000,-0.211900,-0.472900,-0.708700,-0.198700,...,0.000000,0.000000,0.000000,0.163600,0.811500,0.001400,0.000000,0.000000,-0.202700,0.000000
25%,-0.066700,0.000000,-0.036000,-0.099550,0.000000,0.000000,-0.066550,-0.118050,0.000000,-0.052350,...,19.249600,5.164700,8.819950,0.182450,1.149750,0.007550,1.000000,0.000000,-0.099500,0.000000
50%,-0.032500,0.000000,-0.002000,-0.059400,1.000000,0.000000,-0.037200,-0.042400,0.000000,-0.001600,...,23.151200,7.848200,13.342700,0.199600,1.238800,0.010700,1.000000,0.000000,-0.082400,0.000000
75%,0.004850,0.000000,0.029000,-0.017100,2.000000,0.000000,0.002650,0.014300,4.897450,0.056550,...,26.958050,10.683950,19.319450,0.219700,1.325350,0.013900,2.000000,0.000000,-0.066350,0.000000
max,0.141100,4.000000,0.218100,0.129000,6.000000,8.141400,0.249500,0.212200,7.740800,0.168400,...,41.380000,20.805400,27.879600,0.272100,1.548800,0.034600,5.000000,0.429200,-0.007300,2.636100


In [36]:
df.isnull().sum()
df.isnull().sum().sort_values(ascending=False).head(10)


MATS3v         0
VAdjMat        0
MPC10          0
JGT            0
ATS2i          0
CrippenLogP    0
ATS2s          0
ATS2p          0
ATS2v          0
SHBint10       0
dtype: int64

In [39]:
df["Class"].value_counts()

Class
NonToxic    115
Toxic        56
Name: count, dtype: int64

In [40]:
df["Class"].value_counts(normalize=True)

Class
NonToxic    0.672515
Toxic       0.327485
Name: proportion, dtype: float64

In [9]:
# Removing Constant features
from sklearn.feature_selection import VarianceThreshold
constant_filter = VarianceThreshold(threshold=0)
x_constant_removed = constant_filter.fit_transform(x)
x_constant_removed.shape

(171, 1203)

In [10]:
# Removing very low variance features
low_variance_filter = VarianceThreshold(threshold=0.00001)
x_low_var = low_variance_filter.fit_transform(x)
x_low_var.shape

(171, 1189)

In [11]:
# Encoding target variables
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_encoded[:10]


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [12]:
# Scaling of features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_low_var)

In [15]:
# Converting back to dataframe

x_low_var_df = pd.DataFrame(x_low_var)
# Computing correlation using abs since we care about magnitude and not direction

corr_matrix = x_low_var_df.corr().abs()

# Identifing highly correlated pairs (we only need upeer trianle to avoid duplicates)

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]

len(to_drop)

x_uncorr = x_low_var_df.drop(columns=to_drop)

x_uncorr.shape

(171, 472)

In [16]:
# Train/Test split (for final evaluation)

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_uncorr,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [17]:
# Initialization of Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

In [19]:
# Cross validation

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf,
    x_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("F1 scores per fold:", cv_scores)
print("Mean F1:", cv_scores.mean())
rf.fit(x_train, y_train)

F1 scores per fold: [0.         0.28571429 0.16666667 0.18181818 0.36363636]
Mean F1: 0.19956709956709956


RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [20]:
# Evaluating on Test set

from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(x_test)
y_prob = rf.predict_proba(x_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.96      0.81        24
           1       0.50      0.09      0.15        11

    accuracy                           0.69        35
   macro avg       0.60      0.52      0.48        35
weighted avg       0.64      0.69      0.60        35

ROC-AUC: 0.5946969696969697


In [22]:
# Feature Importance pruning

importances = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": x_uncorr.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(20)
(feature_importance_df["importance"] < 0.001).sum()

121

In [23]:
# Running the top 50 

top_k = 50

top_features = feature_importance_df.head(top_k)["feature"]

x_reduced = x_uncorr[top_features]
x_reduced.shape

(171, 50)

In [24]:
# Retraining Model

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_reduced,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [25]:
# Retraining Model

from sklearn.ensemble import RandomForestClassifier

rf_reduced = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_reduced.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [26]:
# Retraining Model

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_reduced,
    x_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Mean CV F1:", cv_scores.mean())

Mean CV F1: 0.29238095238095235


In [27]:
# Evaluation of retrained model

from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf_reduced.predict(x_test)
y_prob = rf_reduced.predict_proba(x_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.69      0.92      0.79        24
           1       0.33      0.09      0.14        11

    accuracy                           0.66        35
   macro avg       0.51      0.50      0.46        35
weighted avg       0.58      0.66      0.58        35

ROC-AUC: 0.6590909090909091


In [28]:
# Adjusting Decison Threshold

y_prob = rf_reduced.predict_proba(x_test)[:, 1]

y_pred_adjusted = (y_prob >= 0.35).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_adjusted))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.72      0.75      0.73        24
           1       0.40      0.36      0.38        11

    accuracy                           0.63        35
   macro avg       0.56      0.56      0.56        35
weighted avg       0.62      0.63      0.62        35

ROC-AUC: 0.6590909090909091
